# ingest

Downloads the AEMO source files into the `dbt_landing` lakehouse. It is the first step of
`run_pipeline` and the only one that lands, so both engines read the same files.

- **What lands**: the daily price and SCADA archive, today's 5-minute SCADA and price files,
  and the DUID reference data, as plain CSV under `Files/csv_raw/`.
- **The archive log**, `Files/csv_raw_archive_log.parquet`, has one row per file landed. It
  is the watermark: a re-run fetches only what is not in it.
- **Newest first**: with a limit smaller than the backlog, the recent end lands first and
  the gap fills backwards over runs.

Settings are in the `deploy_config` Variable Library: `download_limit` (intraday files per
feed per run) and `daily_download_limit` (daily archive files per run).

DuckDB is a library here, for the listings and the log. It ships with the notebook, so
nothing is installed.

In [ ]:
# 1. SETTINGS
import io
import os
import shutil
import tempfile
import time
import urllib.error
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

import duckdb
import notebookutils

vl = notebookutils.variableLibrary.getLibrary("deploy_config")
# Intraday files come straight from nemweb, which throttles a burst with HTTP 403.
download_limit = int(vl.download_limit)
# Daily files are backfilled from a GitHub mirror, so a higher limit is safe.
daily_download_limit = int(vl.daily_download_limit)

# dbt_landing/Files. The lakehouse is this notebook's default one, so it is a local folder.
LANDING = "/lakehouse/default/Files"
LOG = f"{LANDING}/csv_raw_archive_log.parquet"

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs; INSTALL json; LOAD json;")
print(f"Landing to {LANDING}")

In [ ]:
# 2. THE ARCHIVE LOG: one row per file landed. Both engines read it to know what to fold.
if os.path.exists(LOG):
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE _csv_archive_log AS
        SELECT source_type, source_filename, archive_path, archived_at,
               row_count, source_url, etag, csv_filename
        FROM read_parquet('{LOG}') WHERE csv_filename IS NOT NULL""")
else:
    con.sql("""
        CREATE OR REPLACE TEMP TABLE _csv_archive_log (
            source_type VARCHAR, source_filename VARCHAR, archive_path VARCHAR,
            archived_at TIMESTAMPTZ, row_count BIGINT, source_url VARCHAR,
            etag VARCHAR, csv_filename VARCHAR)""")


def log_insert(source_type, source_filename, archive_path, now, url, csv_base):
    """Parameterised: odd characters in a filename or URL cannot break the SQL."""
    con.execute(
        "INSERT INTO _csv_archive_log VALUES (?, ?, ?, CAST(? AS TIMESTAMPTZ), NULL, ?, NULL, ?)",
        [source_type, source_filename, archive_path, now, url, csv_base])


def save_log():
    with tempfile.TemporaryDirectory() as tmp:
        local = os.path.join(tmp, "csv_raw_archive_log.parquet")
        con.sql(f"COPY _csv_archive_log TO '{local}' (FORMAT PARQUET)")
        shutil.copyfile(local, LOG)

In [ ]:
# 3. HOW A FILE LANDS. A ZIP is downloaded, its CSVs are extracted UNCOMPRESSED into a temp
# folder, and the folder is copied into the landing zone. Plain CSV, because Fabric
# Warehouse's OPENROWSET cannot read gzip CSV: that needs PARSER 1.0, and 1.0 cannot parse
# the ragged, quoted AEMO rows.
BATCH_SIZE, MAX_WORKERS = 7, 8


def download_and_extract(url, temp_dir):
    """Download a ZIP and extract its CSVs. Thread-safe."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (dbt-aemo)"})
    for attempt in range(3):
        try:
            zip_bytes = urllib.request.urlopen(req, timeout=60).read()
            break
        except urllib.error.HTTPError:
            if attempt < 2:
                time.sleep(2**attempt)
                continue
            raise
    z = zipfile.ZipFile(io.BytesIO(zip_bytes))
    out = []
    for name in z.namelist():
        if name.upper().endswith(".CSV"):
            safe = name.replace("/", "_")
            with open(os.path.join(temp_dir, safe), "wb") as f:
                f.write(z.read(name))
            out.append(safe)
    return out


def land(local_folder, subfolder, overwrite):
    """Copy a temp folder's files to csv_raw/<subfolder>/. A landed CSV is immutable, so
    overwrite=False keeps the one already there."""
    dest = os.path.join(LANDING, "csv_raw", subfolder)
    os.makedirs(dest, exist_ok=True)
    for name in os.listdir(local_folder):
        if overwrite or not os.path.exists(os.path.join(dest, name)):
            shutil.copyfile(os.path.join(local_folder, name), os.path.join(dest, name))


def process(rows, source_type, subfolder):
    """Land `rows` (url, filename) in batches, logging each batch as it lands."""
    for i in range(0, len(rows), BATCH_SIZE):
        batch = rows[i : i + BATCH_SIZE]
        with tempfile.TemporaryDirectory() as tmp:
            extracted = []
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                futs = {ex.submit(download_and_extract, u, tmp): (u, fn) for u, fn in batch}
                for fut in as_completed(futs):
                    u, fn = futs[fut]
                    try:
                        extracted += [(fn, csv_name, u) for csv_name in fut.result()]
                    except Exception as e:
                        print(f"  WARN skip {fn}: {e}")
            if extracted:
                land(tmp, subfolder, overwrite=False)
            now = datetime.now(timezone.utc).isoformat()
            for fn, csv_name, u in extracted:
                base = csv_name.removesuffix(".CSV").removesuffix(".csv")
                log_insert(source_type, fn, f"/{subfolder}/{csv_name}", now, u, base)
        save_log()


def sql_retry(q, attempts=4, base=5):
    """For the listings. nemweb throws a transient 403/5xx when it rate-limits a burst; back
    off (5s, 10s, 20s) rather than failing the whole run."""
    for a in range(attempts):
        try:
            return con.sql(q)
        except Exception as e:
            if a < attempts - 1:
                w = base * (2**a)
                print(f"  WARN net {type(e).__name__}: {e}; retry {a + 1}/{attempts - 1} in {w}s")
                time.sleep(w)
            else:
                raise


def new_files(table, source_type):
    # ORDER BY filename DESC -- NEWEST FIRST, so a limit smaller than the backlog lands the
    # recent end of the archive and the gap fills backwards over runs. It is the same
    # direction the fact models fold in (macros/new_source_files.sql, spark_new_files.sql).
    limit = daily_download_limit if source_type == "daily" else download_limit
    return con.sql(
        f"""SELECT full_url, filename FROM {table}
            WHERE '{source_type}::' || filename NOT IN (
                SELECT source_type || '::' || source_filename FROM _csv_archive_log)
            ORDER BY filename DESC
            LIMIT {limit}"""
    ).fetchall()

In [ ]:
# 4. DAILY: the price and SCADA records share one file per day.
sql_retry(
    """CREATE OR REPLACE TEMP TABLE daily_files_web AS
    WITH h AS (SELECT content AS html FROM read_text('https://nemweb.com.au/Reports/Current/Daily_Reports/')),
         l AS (SELECT unnest(string_split(html, '<br>')) AS line FROM h)
    SELECT 'https://nemweb.com.au' || regexp_extract(line, 'HREF="([^"]+)"', 1) AS full_url,
           split_part(regexp_extract(line, 'HREF="[^"]+/([^"]+\\.zip)"', 1), '.', 1) AS filename
    FROM l WHERE line LIKE '%PUBLIC_DAILY%.zip%'"""
)
on_nemweb = con.sql(
    """SELECT count(*) FROM daily_files_web
       WHERE 'daily::' || filename NOT IN (
           SELECT source_type || '::' || source_filename FROM _csv_archive_log)"""
).fetchone()[0]

if on_nemweb < daily_download_limit:
    # nemweb only keeps the current window, so history comes from a GitHub mirror.
    years = range(2018, datetime.now(timezone.utc).year + 1)
    reads = " UNION ALL ".join(
        "SELECT content AS j FROM read_text("
        f"'https://api.github.com/repos/djouallah/aemo_data/contents/data/archive/{y}')"
        for y in years
    )
    # Opportunistic: if the listing is rate-limited or down, carry on with whatever nemweb
    # is serving rather than failing the run.
    try:
        sql_retry(
            f"""INSERT INTO daily_files_web
            WITH api AS ({reads}),
                 p AS (SELECT unnest(from_json(j, '["json"]')) AS fi FROM api)
            SELECT json_extract_string(fi, '$.download_url') AS full_url,
                   split_part(json_extract_string(fi, '$.name'), '.', 1) AS filename
            FROM p WHERE json_extract_string(fi, '$.name') LIKE 'PUBLIC_DAILY%.zip'
              AND split_part(json_extract_string(fi, '$.name'), '.', 1)
                  NOT IN (SELECT filename FROM daily_files_web)"""
        )
    except Exception as e:
        print(f"  WARN: GitHub backfill listing unavailable, continuing with nemweb only: {e}")

daily = new_files("daily_files_web", "daily")
process(daily, "daily", "daily")
print(f"daily: {len(daily)} files")

In [ ]:
# 5. INTRADAY SCADA: today's 5-minute files, which the daily archive does not hold yet.
sql_retry(
    """CREATE OR REPLACE TEMP TABLE intraday_scada_web AS
    WITH h AS (SELECT content AS html FROM read_text('http://nemweb.com.au/Reports/Current/Dispatch_SCADA/')),
         l AS (SELECT unnest(string_split(html, '<br>')) AS line FROM h)
    SELECT 'http://nemweb.com.au' || regexp_extract(line, 'HREF="([^"]+)"', 1) AS full_url,
           split_part(regexp_extract(line, 'HREF="[^"]+/([^"]+\\.zip)"', 1), '.', 1) AS filename
    FROM l WHERE line LIKE '%PUBLIC_DISPATCHSCADA%' ORDER BY full_url DESC LIMIT 500"""
)
scada = new_files("intraday_scada_web", "scada_today")
process(scada, "scada_today", "scada_today")
print(f"scada_today: {len(scada)} files")

In [ ]:
# 6. INTRADAY PRICE
sql_retry(
    """CREATE OR REPLACE TEMP TABLE intraday_price_web AS
    WITH h AS (SELECT content AS html FROM read_text('http://nemweb.com.au/Reports/Current/DispatchIS_Reports/')),
         l AS (SELECT unnest(string_split(html, '<br>')) AS line FROM h)
    SELECT 'http://nemweb.com.au' || regexp_extract(line, 'HREF="([^"]+)"', 1) AS full_url,
           split_part(regexp_extract(line, 'HREF="[^"]+/([^"]+\\.zip)"', 1), '.', 1) AS filename
    FROM l WHERE line LIKE '%PUBLIC_DISPATCHIS_%.zip%' ORDER BY full_url DESC LIMIT 500"""
)
price = new_files("intraday_price_web", "price_today")
process(price, "price_today", "price_today")
print(f"price_today: {len(price)} files")

In [ ]:
# 7. DUID REFERENCE DATA: the generating units. Small, replaced whole, at most once a day.
duid_sources = [
    ("duid_data", "duid_data",
     "https://raw.githubusercontent.com/djouallah/aemo_data/refs/heads/main/duid_data.csv",
     "duid_data.csv"),
    ("duid_facilities", "facilities",
     "https://data.wa.aemo.com.au/datafiles/post-facilities/facilities.csv",
     "facilities.csv"),
    ("duid_wa_energy", "WA_ENERGY",
     "https://raw.githubusercontent.com/djouallah/aemo_data/refs/heads/main/WA_ENERGY.csv",
     "WA_ENERGY.csv"),
    ("duid_geo_data", "geo_data",
     "https://raw.githubusercontent.com/djouallah/aemo_data/refs/heads/main/geo_data.csv",
     "geo_data.csv"),
]
last = con.sql(
    "SELECT max(archived_at) FROM _csv_archive_log WHERE source_type LIKE 'duid_%'"
).fetchone()[0]
if last is not None and (datetime.now(last.tzinfo) - last).total_seconds() < 86400:
    print(f"DUID data fresh ({last}), skipping")
else:
    with tempfile.TemporaryDirectory() as tmp:
        for source_type, source_filename, url, csv_name in duid_sources:
            header = ", header=true" if source_filename == "WA_ENERGY" else ""
            con.sql(
                f"""COPY (SELECT * FROM read_csv_auto('{url}',
                          null_padding=true, ignore_errors=true{header}))
                    TO '{os.path.join(tmp, csv_name)}' (FORMAT CSV, HEADER)"""
            )
        land(tmp, "duid", overwrite=True)
    con.sql("DELETE FROM _csv_archive_log WHERE source_type LIKE 'duid_%'")
    now = datetime.now(timezone.utc).isoformat()
    for source_type, source_filename, url, csv_name in duid_sources:
        log_insert(source_type, source_filename, f"/duid/{csv_name}", now, url,
                   csv_name.rsplit(".", 1)[0])

In [ ]:
# 8. WHAT IS LANDED NOW
save_log()
for source_type, files in con.sql(
    """SELECT source_type, count(*) AS files FROM _csv_archive_log
       GROUP BY source_type ORDER BY source_type"""
).fetchall():
    print(f"  {source_type:<16}{files:>6} files")